# One-beam 2-D electric-field reconstruction

This notebook runs the $S=1/64$ reflected-beam test from Sec. III B of Follett *et al.*, *Physics of Plasmas* **29**, 113902 (2022), in two equivalent hydro coordinate systems:

- a Cartesian $x$-$y$ grid;
- a cylindrical $r$-$\phi$ grid with a 1 m reference length in $z$.

A single 351 nm super-Gaussian beam propagates in $+x$ through the circular LILAC density profile, refracts, forms a caustic envelope, and reflects. Each trace remains natively three-dimensional with zero $z$ direction. The two ray sheets are triangulated and coherently summed after the reflected sheet receives its $-\pi/2$ caustic phase. Inverse bremsstrahlung is disabled.

> This demonstrates the project's improved field-limiter reconstruction. It does not implement the etalon-integral or LPSE solutions also shown in the paper.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd()
if not (repo_root / "configs").is_dir():
    repo_root = repo_root.parent
if not (repo_root / "configs").is_dir():
    raise RuntimeError("Run this notebook from the repository root or examples folder")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from examples._workflows import (
    TWO_DIMENSIONAL_FIELD_CONFIGS,
    run_two_dimensional_field_reconstruction,
)
from pyGATH.raytracing import RAY_SHEET_LAYOUT, RAY_STATE_LAYOUT, critical_density

In [ ]:
cases = {}
for geometry, default_path in TWO_DIMENSIONAL_FIELD_CONFIGS.items():
    config_path = repo_root / "configs" / "example_configs" / default_path.name
    started = time.perf_counter()
    cases[geometry] = run_two_dimensional_field_reconstruction(config_path)
    elapsed = time.perf_counter() - started
    result, grid, beams, reconstruction, checks = cases[geometry]
    print(
        f"{geometry:11s}: {elapsed:6.2f} s, grid={grid.ncells[:2]}, caustic rays={checks['rays_with_caustics']}/{checks['total_rays']}, max caustic ne/ncrit={checks['maximum_caustic_density_over_ncritical']:.6f}"
    )

## Density and ray sheets

Both decks evaluate the same circular profile, $n_e/n_c=1.165(d/r)^{3.78}$ with $d=343S\,\mu\mathrm{m}$ and $S=1/64$. Density is capped at $4n_c$ inside the critical circle solely to regularize the unresolved $r=0$ singularity. It cannot affect rays because they turn at or below $n_e/n_c=1$.

In [ ]:
def plot_native(ax, reconstruction, values, **kwargs):
    boundaries = reconstruction["cartesian_boundaries_m"] * 1e6
    artist = ax.pcolormesh(
        boundaries[..., 0],
        boundaries[..., 1],
        values,
        shading="flat",
        rasterized=True,
        **kwargs,
    )
    ax.set_aspect("equal")
    ax.set_xlabel(r"$x$ [$\mu$m]")
    ax.set_ylabel(r"$y$ [$\mu$m]")
    ax.set_xlim(-16, 16)
    ax.set_ylim(-16, 16)
    return artist


critical_radius_um = 343.0 / 64.0 * 1.165 ** (1.0 / 3.78)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), constrained_layout=True)
for ax, (geometry, case) in zip(axes, cases.items(), strict=True):
    result, grid, beams, reconstruction, checks = case
    centres = reconstruction["cartesian_centres_m"]
    hydro = grid.interpolate(centres.reshape(-1, 3))
    density_ratio = np.asarray(hydro.ne).reshape(grid.ncells[:2]) / float(
        critical_density(beams.omega[0])
    )
    image = plot_native(
        ax, reconstruction, density_ratio, cmap="magma", vmin=0.0, vmax=1.2
    )
    positions = np.asarray(result.sheet_fields)[0, ..., RAY_STATE_LAYOUT.position]
    for sheet in range(2):
        for ray in range(0, positions.shape[1], 4):
            ax.plot(
                positions[sheet, ray, 0, :, 0] * 1e6,
                positions[sheet, ray, 0, :, 1] * 1e6,
                color="white",
                alpha=0.32,
                linewidth=0.45,
            )
    ax.add_patch(
        plt.Circle(
            (0, 0),
            critical_radius_um,
            fill=False,
            color="cyan",
            linestyle="--",
            linewidth=1.0,
        )
    )
    ax.set_title(f"{geometry}: density and ray sheets")
fig.colorbar(image, ax=axes, label=r"$n_e/n_{crit}$", shrink=0.86);

## Coherent field on the native cell centres

For each geometry, the ray lattice and path samples form a triangular field on each sheet. The physical uncapped field is formed at every ray-sheet vertex before interpolation; this is essential because the initial electric field varies across the transverse super-Gaussian profile. Phase length and field amplitude are then linearly interpolated to the grid cell centres before applying

$$E(x,y)=\sum_j |E_j|\exp\left[i\left(\frac{\omega}{c}\ell_{\phi,j}-\frac{\pi\alpha_j}{2}\right)\right],$$

where $\alpha=0$ on the incident sheet and $\alpha=1$ on the reflected sheet. The plots below are expressed in Cartesian physical space even when the underlying storage is $r$-$\phi$, and all fields use the absolute dimensionless quiver amplitude $a=eE/(m_e c\omega_0)$ rather than normalization to a sampled ray.

In [ ]:
normalized_fields = {}
real_samples = []
magnitude_samples = []
for geometry, (_result, _grid, _beams, reconstruction, _checks) in cases.items():
    to_quiver_amplitude = reconstruction["quiver_amplitude_per_v_m"]
    field = reconstruction["capped_field_v_m"] * to_quiver_amplitude
    active = np.any(reconstruction["inside"], axis=0)
    normalized_fields[geometry] = (field, active)
    real_samples.append(np.real(field[active]))
    magnitude_samples.append(np.abs(field[active]))
real_limit = np.percentile(np.abs(np.concatenate(real_samples)), 99.5)
magnitude_limit = np.percentile(np.concatenate(magnitude_samples), 99.5)

fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
for column, (geometry, case) in enumerate(cases.items()):
    reconstruction = case[3]
    field, active = normalized_fields[geometry]
    real_field = np.where(active, np.real(field), np.nan)
    magnitude = np.where(active, np.abs(field), np.nan)
    real_image = plot_native(
        axes[0, column],
        reconstruction,
        real_field,
        cmap="RdBu_r",
        vmin=-real_limit,
        vmax=real_limit,
    )
    magnitude_image = plot_native(
        axes[1, column],
        reconstruction,
        magnitude,
        cmap="viridis",
        vmin=0.0,
        vmax=magnitude_limit,
    )
    axes[0, column].set_title(
        rf"{geometry}: coherent $e\,\mathrm{{Re}}(E)/(m_e c\omega_0)$"
    )
    axes[1, column].set_title(rf"{geometry}: coherent $e|E|/(m_e c\omega_0)$")
fig.colorbar(
    real_image, ax=axes[0], label=r"$e\,\mathrm{Re}(E)/(m_e c\omega_0)$", shrink=0.82
)
fig.colorbar(magnitude_image, ax=axes[1], label=r"$e|E|/(m_e c\omega_0)$", shrink=0.82);

## Common Cartesian lineout and caustic limiting

The helper also evaluates both triangular fields at the same 1,400 Cartesian points on $y=0$. This avoids comparing different native cell locations. On each individual sheet the uncapped magnitude must be greater than or equal to the capped magnitude. That ordering does **not** necessarily hold for the magnitude of the coherent two-sheet sum: changing the relative incident/reflected amplitudes can strengthen or weaken destructive interference. The lower panels expose the individual sheets separately from that interference effect.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), constrained_layout=True)
for column, (geometry, (_result, _grid, _beams, reconstruction, _checks)) in enumerate(
    cases.items()
):
    line = reconstruction["lineout"]
    x_um = line["x_m"] * 1e6
    to_quiver_amplitude = reconstruction["quiver_amplitude_per_v_m"]
    capped = line["capped_field_v_m"] * to_quiver_amplitude
    uncapped = line["uncapped_field_v_m"] * to_quiver_amplitude
    if geometry == "cartesian":
        ls = "-"
        lw = 1.5
    else:
        ls = "--"
        lw = 2.0
    axes[0, 0].plot(x_um, np.real(capped), label=geometry, linestyle=ls, linewidth=lw)
    axes[0, 1].plot(
        x_um, np.abs(uncapped), linestyle=":", alpha=0.65, label=f"{geometry} uncapped"
    )
    axes[0, 1].plot(
        x_um, np.abs(capped), label=f"{geometry} capped", linestyle=ls, linewidth=lw
    )
    for sheet in range(2):
        uncapped_sheet = (
            line["uncapped_sheet_magnitude_v_m"][sheet] * to_quiver_amplitude
        )
        capped_sheet = line["capped_sheet_magnitude_v_m"][sheet] * to_quiver_amplitude
        axes[1, column].plot(
            x_um,
            uncapped_sheet,
            linestyle=":",
            alpha=0.6,
            label=f"sheet {sheet + 1} uncapped",
        )
        axes[1, column].plot(x_um, capped_sheet, label=f"sheet {sheet + 1} capped")
    axes[1, column].set_title(f"{geometry}: individual sheet magnitudes")
for ax in axes.flat:
    ax.axvline(-critical_radius_um, color="black", linestyle="--", linewidth=0.8)
    ax.set_xlim(-7.5, -5.2)
    ax.set_xlabel(r"$x$ [$\mu$m]")
axes[0, 0].set(
    ylabel=r"$e\,\mathrm{Re}(E)/(m_e c\omega_0)$",
    title="Common $y=0$ coherent quiver field",
)
axes[0, 1].set(
    ylabel=r"$e|E|/(m_e c\omega_0)$", title="Coherent magnitude: capped versus uncapped"
)
axes[1, 0].set_ylabel(r"$e|E_j|/(m_e c\omega_0)$")
axes[1, 1].set_ylabel(r"$e|E_j|/(m_e c\omega_0)$")
for ax in axes.flat:
    ax.legend(fontsize=8)

In [ ]:
for geometry, (result, grid, _beams, reconstruction, checks) in cases.items():
    active = np.any(reconstruction["inside"], axis=0)
    assert grid.dimensions == 2
    assert checks["maximum_inverse_brems_deposition_w_m3"] == 0.0
    assert checks["maximum_caustic_density_over_ncritical"] > 0.95
    assert np.all(np.isfinite(reconstruction["capped_field_v_m"][active]))
    valid_sheet = reconstruction["inside"]
    uncapped_sheet = reconstruction["uncapped_sheet_magnitude_v_m"]
    capped_sheet = reconstruction["capped_sheet_magnitude_v_m"]
    tolerance = 1e-12 * np.max(uncapped_sheet[valid_sheet])
    assert np.all(uncapped_sheet[valid_sheet] + tolerance >= capped_sheet[valid_sheet])
    fields = np.asarray(result.sheet_fields)
    assert np.max(fields[..., RAY_SHEET_LAYOUT.uncapped_amplitude]) > np.max(
        fields[..., RAY_SHEET_LAYOUT.capped_amplitude]
    )
    assert np.allclose(fields[..., RAY_STATE_LAYOUT.position][..., 2], 0.0)
    assert np.allclose(fields[..., RAY_STATE_LAYOUT.momentum][..., 2], 0.0)
print("Cartesian and cylindrical 2-D reconstruction checks passed.")